# Provider-Vergleich

Fuehrt denselben Artikel automatisiert gegen mehrere konfigurierte LLM-Provider aus `llm_adapter` und vergleicht die Ergebnisse — formalisiert die Wegwerf-Skripte, die diese Session einzeln fuer Qwen/Mistral/OpenAI/Gemini geschrieben wurden.

**Voraussetzungen:**
- ChromaDB laeuft lokal (`chroma run --host localhost --port 8001 --path ../data/chroma_db`) — `analyze_article()` braucht es fuer Anker/Normalisierung.
- Fuer jeden getesteten Provider muss der jeweilige Key/lokale Server bereitstehen (siehe [`docs/environments/local.md`](../docs/environments/local.md)).
- Jeder Provider-Fehlschlag (Quota, Rate-Limit, fehlender Key) wird abgefangen und im Ergebnis als Fehler markiert, statt das ganze Notebook abzubrechen.

## 1 — Setup

In [ ]:
import os, sys, time, traceback
sys.path.insert(0, "../src")

from news_analyser.scraper import Article
from news_analyser.agents.analyzer import analyze_article

print("Setup OK")

## 2 — Artikel & Provider waehlen

`TEXT_SOURCE` kann ein Pfad zu einer Debug-Datei sein (Standard: letzter Lauf) oder direkt Artikeltext. `PROVIDERS` ist die Liste der zu testenden `LLM_PROVIDER`-Werte — muss in `llm_adapter`/`news_analyser/__init__.py` registriert sein.

In [ ]:
TEXT_SOURCE = "../data/debug_last_run/00_original_text.txt"
ARTICLE_URL = "https://www.derstandard.at/story/3000000333805/digitaltrainer-kinder-koennten-ki-bald-besser-finden-als-menschen?ref=rss"
ARTICLE_DOMAIN = "derstandard.at"
ARTICLE_TITLE = 'Digitaltrainer: "Kinder koennten KI bald besser finden als Menschen"'

PROVIDERS = ["gemini", "mistral", "lm_studio", "openai"]

import os as _os
if _os.path.isfile(TEXT_SOURCE):
    article_text = open(TEXT_SOURCE, encoding="utf-8").read()
else:
    article_text = TEXT_SOURCE

print(f"{len(article_text.split())} Woerter geladen")

## 3 — Laeufe ausfuehren

Ein Call pro Provider, sequentiell (parallel wuerde ChromaDB-Writes/Anker durcheinanderbringen).

In [ ]:
def run_provider(provider: str) -> dict:
    os.environ["LLM_PROVIDER"] = provider
    article = Article(
        url=ARTICLE_URL, domain=ARTICLE_DOMAIN, text=article_text,
        fetched_at="2026-09-16T08:14:01.323147Z", title=ARTICLE_TITLE, author="",
        published_at="2026-09-16T08:14:01.323147Z", word_count=len(article_text.split()),
    )
    t0 = time.time()
    try:
        result = analyze_article(article)
        dt = time.time() - t0
        if result is None:
            return {"provider": provider, "status": "failed (None)", "duration_s": round(dt, 1)}
        ft = result["framing_target"]
        techs = result.get("detected_techniques", [])
        pwc = result.get("pass1_word_count") or article.word_count
        stroemung = [s.get("label") if isinstance(s, dict) else s for s in result.get("politische_stroemung", [])]
        return {
            "provider": provider, "status": "ok", "duration_s": round(dt, 1),
            "model": result.get("llm_model"),
            "orwell_structural": ft.get("orwell_index_structural"),
            "quote_amplification": ft.get("quote_amplification_index"),
            "orwell_index": ft.get("orwell_index"),
            "bernays_score": round(len(techs) / max(pwc, 1) * 1000, 2),
            "technique_count": len(techs),
            "dk_index": ft.get("dunning_kruger_index"),
            "stroemung": ", ".join(stroemung),
        }
    except Exception as exc:
        dt = time.time() - t0
        return {"provider": provider, "status": f"error: {exc}", "duration_s": round(dt, 1)}

results = []
for p in PROVIDERS:
    print(f"--- {p} ---", flush=True)
    r = run_provider(p)
    print(f"  {r['status']}  ({r['duration_s']}s)")
    results.append(r)

## 4 — Vergleichstabelle

In [ ]:
import pandas as pd

df = pd.DataFrame(results).set_index("provider")
pd.set_option("display.max_colwidth", 60)
df